# Module 02 — Solutions: Calculus & Gradient Descent

In [1]:
import numpy as np
rng = np.random.default_rng(7)

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

def numerical_gradient(f, p, h=1e-5):
    grad = np.zeros_like(p, dtype=float)
    for i in range(len(p)):
        step = np.zeros_like(p, dtype=float)
        step[i] = h
        grad[i] = (f(p + step) - f(p - step)) / (2 * h)
    return grad

### Exercise 1: Derivatives by hand
Write the derivative of each function as a Python function. The checks compare your answers with numerical derivatives.

1. $f(x) = 4x^3 - 2x + 7$
2. $g(x) = e^{2x}$ (chain rule: the inner function is 2x)
3. $h(x) = \ln(1 + x^2)$ (chain rule)

In [2]:
df = lambda x: 12 * x ** 2 - 2
dg = lambda x: 2 * np.exp(2 * x)
dh = lambda x: 2 * x / (1 + x ** 2)

In [3]:
# check
f = lambda x: 4 * x ** 3 - 2 * x + 7
g = lambda x: np.exp(2 * x)
h = lambda x: np.log(1 + x ** 2)
for x0 in [-1.3, 0.0, 0.7, 2.0]:
    assert np.isclose(df(x0), numerical_derivative(f, x0), rtol=1e-4), "check df"
    assert np.isclose(dg(x0), numerical_derivative(g, x0), rtol=1e-4), "check dg"
    assert np.isclose(dh(x0), numerical_derivative(h, x0), rtol=1e-4, atol=1e-6), "check dh"
print("✅ Exercise 1 passed")

✅ Exercise 1 passed


### Exercise 2: A gradient by hand
For $f(x, y) = x^2 y + 3y^2 - 4x$, implement `grad_f(p)`, which returns $[\partial f/\partial x,\ \partial f/\partial y]$.

In [4]:
def f_xy(p):
    x, y = p
    return x ** 2 * y + 3 * y ** 2 - 4 * x

def grad_f(p):
    x, y = p
    return np.array([2 * x * y - 4, x ** 2 + 6 * y])

In [5]:
# check
for _ in range(5):
    p = rng.normal(size=2) * 3
    assert np.allclose(grad_f(p), numerical_gradient(f_xy, p), rtol=1e-4, atol=1e-6)
print("✅ Exercise 2 passed")

✅ Exercise 2 passed


### Exercise 3: Implement gradient descent
Implement `gradient_descent(grad_fn, p_start, lr, n_steps)`. It should return the **final** parameters.
Then use it to find the minimum of
$$ q(a, b) = (a - 2)^2 + (b + 1)^2 + 0.5\,a b $$
Derive `grad_q` yourself.

In [6]:
def gradient_descent(grad_fn, p_start, lr, n_steps):
    p = np.array(p_start, dtype=float)
    for _ in range(n_steps):
        p = p - lr * grad_fn(p)
    return p

def grad_q(p):
    a, b = p
    return np.array([2 * (a - 2) + 0.5 * b, 2 * (b + 1) + 0.5 * a])

p_min = gradient_descent(grad_q, [0.0, 0.0], lr=0.1, n_steps=500)

In [7]:
# check: at the minimum the gradient is ~0
q = lambda p: (p[0] - 2) ** 2 + (p[1] + 1) ** 2 + 0.5 * p[0] * p[1]
assert np.allclose(numerical_gradient(q, p_min), 0, atol=1e-5)
assert np.allclose(p_min, [2.4, -1.6], atol=1e-4), p_min
print("✅ Exercise 3 passed ->", p_min)

✅ Exercise 3 passed -> [ 2.4 -1.6]


### Exercise 4: Learning-rate experiment
Using your `gradient_descent` and `grad_q`, run 50 steps from `[0, 0]` for each learning rate below.
Fill `final_loss[lr]` with `q(final_p)`. Then work out which rates converged and which diverged.

In [8]:
lrs = [0.001, 0.05, 0.5, 1.2]
final_loss = {}
for lr in lrs:
    final_loss[lr] = q(gradient_descent(grad_q, [0.0, 0.0], lr=lr, n_steps=50))
final_loss

{0.001: np.float64(4.07508914761412),
 0.05: np.float64(-1.397531482105172),
 0.5: np.float64(-1.4),
 1.2: np.float64(5.070602400912896e+29)}

In [9]:
# check
best = q(np.array([2.4, -1.6]))
assert final_loss[0.001] > best + 1, "lr=0.001 should still be far from the minimum after 50 steps"
assert np.isclose(final_loss[0.05], best, atol=1e-2)
assert np.isclose(final_loss[0.5], best, atol=1e-6)
assert final_loss[1.2] > 1e6, "lr=1.2 should diverge"
print("✅ Exercise 4 passed")

✅ Exercise 4 passed


### Exercise 5: Fit a line with gradient descent
Write `fit_line(x, y, lr, n_steps)`, which minimizes the MSE of `y ≈ w*x + b` with gradient descent, starting from
`w = b = 0`. Return `(w, b)`.

In [10]:
def fit_line(x, y, lr=0.01, n_steps=3000):
    w, b = 0.0, 0.0
    for _ in range(n_steps):
        error = w * x + b - y
        w -= lr * 2 * np.mean(error * x)
        b -= lr * 2 * np.mean(error)
    return w, b

x = rng.uniform(-3, 3, 300)
y = -1.7 * x + 4.2 + rng.normal(0, 0.3, 300)
w_fit, b_fit = fit_line(x, y)

In [11]:
# check against the closed-form least-squares answer
w_ref, b_ref = np.polyfit(x, y, deg=1)
assert np.isclose(w_fit, w_ref, atol=1e-3) and np.isclose(b_fit, b_ref, atol=1e-3), (w_fit, b_fit)
print(f"✅ Exercise 5 passed -> w={w_fit:.3f}, b={b_fit:.3f}")

✅ Exercise 5 passed -> w=-1.712, b=4.154


### Exercise 6 (challenge): Fit a parabola
The data follows $y = a x^2 + b x + c$. Extend your gradient descent to learn all three parameters.
Derive the three partial derivatives of the MSE (they have the same pattern as for the line).

In [12]:
x = rng.uniform(-2, 2, 400)
y = 0.8 * x ** 2 - 1.5 * x + 0.5 + rng.normal(0, 0.1, 400)

a = b = c = 0.0
lr = 0.05
for _ in range(5000):
    error = a * x ** 2 + b * x + c - y
    a -= lr * 2 * np.mean(error * x ** 2)
    b -= lr * 2 * np.mean(error * x)
    c -= lr * 2 * np.mean(error)

In [13]:
# check
ref = np.polyfit(x, y, deg=2)
assert np.allclose([a, b, c], ref, atol=1e-3), ([a, b, c], ref)
print("✅ Exercise 6 passed ->", np.round([a, b, c], 3))

✅ Exercise 6 passed -> [ 0.8   -1.504  0.498]
